# Lab 25: AI Literacy & Generative Tools — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 45 min Portfolio + 45 min RAG Pipeline

---

**Format:** Part 1 builds a portfolio site with Claude Code (faster-paced than 3916). Part 2 presents a **deliberately broken RAG pipeline** that you must diagnose, fix, and extend.

**Learning Objectives:**
- Deploy a professional portfolio website using AI-assisted development
- Diagnose configuration errors in a Retrieval-Augmented Generation pipeline
- Compare RAG-grounded answers to direct LLM answers on economic questions
- Evaluate retrieval quality with precision and relevance metrics

**Tools:** Claude Code CLI, Next.js, Vercel, LangChain, ChromaDB, OpenAI API

**Time estimate:** ~90 minutes total

---

---

## Part 1: Portfolio Site (45 min)

Build and deploy a professional portfolio site with Claude Code. You are expected to move quickly — the setup and scaffold steps are compressed.

### Setup (5 min)

```bash
# Install Claude Code CLI (if not already installed)
npm install -g @anthropic-ai/claude-code
claude login
claude skill install /build-portfolio-site
```

Create a [Vercel account](https://vercel.com) via GitHub SSO if you do not have one.

### Scaffold & Customize (25 min)

```bash
mkdir econ-portfolio && cd econ-portfolio
claude
```

In the Claude Code session, provide your resume content, style preferences (browse [stitch.withgoogle.com](https://stitch.withgoogle.com) for reference), and at least **3 projects** from your coursework. Your prompt should include:

- Name, program, professional bio
- Style reference (colors, layout preference)
- Project titles, descriptions, GitHub links, and technologies
- A custom section (choose: Data Projects gallery, Research Interests, or Skills display)

### Deploy (10 min)

```bash
git init && git add . && git commit -m "Initial portfolio"
gh repo create econ-portfolio --public --push
```

Import to Vercel at [vercel.com/new](https://vercel.com/new). Test on desktop and mobile.

### Reflection (5 min)

In a markdown cell below, briefly answer:
1. What prompts worked best? What required iteration?
2. What did you have to fix that the AI got wrong?
3. How does this compare to writing the site from scratch?

**Your live URL:** https://vercel.com/zackdihels-projects/econ-portfolio/CvRxNvEpxP9eijMjr9ADqHamVyGA

*Part 1 Reflection:*

1. My prompt providing resume/experience information worked exactly how I was hoping, Claude translated it into a legible resume webpage very easily.
2. Some of the terminal commands had trouble. There were packages and libraries I did not have installed or pathed correctly and had to troubleshoot in order to get operating properly.
3. Writing a resume from scratch is something I'm experienced with, but creating a website for that information is not something I've ever done before, so I imagine using Claude in this way was a lot easier.

---

## Part 2: RAG Pipeline — Diagnose, Fix, Extend (45 min)

The code below implements a Retrieval-Augmented Generation (RAG) pipeline that ingests FOMC minutes (from your Ch 23 lab) and answers economic questions grounded in those documents.

**The pipeline has 3 deliberate errors:**
1. A **chunking configuration** error
2. An **embedding model** error
3. A **retrieval parameter** error

Your job: run the code, identify each error, explain why it matters, and fix it.

In [3]:
import sys
!{sys.executable} -m pip install langchain langchain-community langchain-openai chromadb tiktoken -q
!pip install openai

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Install required packages
# -----------------------------------------------------------

import os
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
#from langchain_community.chains import RetrievalQA
from langchain_core.documents import Document


# Set your OpenAI API key
os.environ['OPENAI_API_KEY'] = ''

print('Libraries loaded. Ready to diagnose.')

ImportError: cannot import name 'RetrievalQA' from 'langchain_community.chains' (/Users/zacharydihel/Documents/GitHub/ECON-5200-Data-Analytics/venv/lib/python3.14/site-packages/langchain_community/chains/__init__.py)

In [4]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Load 5 sample FOMC minutes (simulated for lab purposes)
# In your Ch 23 lab you scraped real FOMC minutes; here we
# use representative excerpts for reproducibility.
# -----------------------------------------------------------

fomc_documents = [
    Document(
        page_content="""Federal Open Market Committee - January 2023 Meeting.
        Participants noted that inflation remained well above the Committee's
        2 percent objective. The labor market remained very tight, with the
        unemployment rate near historical lows. Wage growth had remained
        elevated relative to what would be consistent with 2 percent inflation
        over time given prevailing trends in productivity growth. Participants
        agreed that the Committee should continue to raise the target range
        for the federal funds rate at upcoming meetings. Several participants
        noted the risk that the lagged effects of cumulative policy tightening
        could end up being more restrictive than necessary.""",
        metadata={"meeting": "January 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - March 2023 Meeting.
        Recent banking sector developments were likely to result in tighter
        credit conditions for households and businesses. Participants
        discussed the effects of the recent banking stress on the economic
        outlook, noting that tighter credit conditions would likely weigh on
        economic activity. Some participants noted that monetary policy
        actions and banking stress were working in the same direction to slow
        the economy. Inflation remained elevated, and recent data provided
        few signs that inflationary pressures were abating quickly enough.""",
        metadata={"meeting": "March 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - June 2023 Meeting.
        Participants noted that the pace of job gains had slowed but remained
        solid. Consumer spending appeared to have picked up. The housing sector
        remained weak, reflecting higher mortgage rates. Participants expected
        inflation to come down as the effects of tight monetary policy worked
        through the economy, though the process was expected to take time.
        Participants generally judged that, with inflation still well above
        the Committee's longer-run goal of 2 percent, keeping the federal
        funds rate at a restrictive level was appropriate.""",
        metadata={"meeting": "June 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - September 2023 Meeting.
        Participants discussed the uncertainty surrounding the economic outlook
        and noted various risks. Supply-side improvements had contributed to
        the decline in inflation. Labor market conditions had eased somewhat
        but remained tight. Several participants emphasized the importance of
        communicating that the Committee would proceed carefully in determining
        the extent of additional policy firming. The Committee decided to
        maintain the target range for the federal funds rate at 5-1/4 to
        5-1/2 percent.""",
        metadata={"meeting": "September 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - December 2023 Meeting.
        Participants noted that the disinflation process was continuing.
        The labor market remained strong with solid job gains and the
        unemployment rate remaining low. Several participants pointed to
        the risk that progress on inflation could stall. The median
        projection for the federal funds rate at end of 2024 was 4.6 percent,
        suggesting rate cuts could begin in 2024. Participants emphasized
        that the timing and pace of rate cuts would depend on incoming data
        and the evolving economic outlook.""",
        metadata={"meeting": "December 2023", "year": 2023}
    ),
]

print(f'Loaded {len(fomc_documents)} FOMC minutes documents.')
for doc in fomc_documents:
    print(f"  - {doc.metadata['meeting']}: {len(doc.page_content)} characters")

Loaded 5 FOMC minutes documents.
  - January 2023: 731 characters
  - March 2023: 649 characters
  - June 2023: 649 characters
  - September 2023: 617 characters
  - December 2023: 609 characters


### DIAGNOSE: Error 1 — Chunking Configuration

The text splitter below has **two problems** with its configuration. Run the cell and examine the output to identify them.

In [5]:
# -----------------------------------------------------------
# DIAGNOSE: This code has errors in the chunking configuration.
# -----------------------------------------------------------

# ERROR 1a: chunk_size is far too small — 50 characters means each
# chunk is just a few words, destroying semantic coherence.
# A good chunk_size for RAG is typically 500-1000 characters.
#
# ERROR 1b: chunk_overlap is 0 — sentences at chunk boundaries
# get split in the middle, losing context. A typical overlap
# is 10-20% of chunk_size (e.g., 100 for chunk_size=500).

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,        # BUG: way too small!
    chunk_overlap=0,      # BUG: no overlap means lost context at boundaries
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(fomc_documents)

print(f'Number of chunks: {len(chunks)}')
print(f'Average chunk length: {np.mean([len(c.page_content) for c in chunks]):.0f} characters')
print(f'\nFirst 5 chunks:')
for i, chunk in enumerate(chunks[:5]):
    print(f'  Chunk {i}: "{chunk.page_content[:80]}..." ({len(chunk.page_content)} chars)')

print(f'\nPROBLEM: With 50-character chunks, each chunk is just a sentence fragment.')
print('The retriever will match on fragments, not meaningful passages.')

Number of chunks: 97
Average chunk length: 29 characters

First 5 chunks:
  Chunk 0: "Federal Open Market Committee - January 2023..." (44 chars)
  Chunk 1: "Meeting...." (8 chars)
  Chunk 2: "Participants noted that inflation..." (33 chars)
  Chunk 3: "remained well above the Committee's..." (35 chars)
  Chunk 4: "2 percent objective..." (19 chars)

PROBLEM: With 50-character chunks, each chunk is just a sentence fragment.
The retriever will match on fragments, not meaningful passages.


### DIAGNOSE: Error 2 — Embedding Model Mismatch

The embedding model below is wrong for this task. Identify the issue.

In [6]:
# -----------------------------------------------------------
# DIAGNOSE: The embedding model is wrong.
# -----------------------------------------------------------

# ERROR 2: Using text-embedding-ada-002 which is the legacy/deprecated
# model. The current best model is text-embedding-3-small (or 3-large).
# ada-002 produces 1536-dim embeddings with lower quality on retrieval
# benchmarks compared to text-embedding-3-small (which also supports
# dimensionality reduction via the 'dimensions' parameter).

embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"  # BUG: deprecated model
)

# Create vector store from (badly chunked) documents
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="fomc_minutes_broken"
)

print(f'Vector store created with {vectorstore._collection.count()} vectors.')
print(f'Embedding model: text-embedding-ada-002 (deprecated)')
print(f'Embedding dimensions: 1536')

Vector store created with 97 vectors.
Embedding model: text-embedding-ada-002 (deprecated)
Embedding dimensions: 1536


### DIAGNOSE: Error 3 — Retrieval Parameter

The retriever configuration has a parameter that makes retrieval ineffective.

In [17]:
# -----------------------------------------------------------
# DIAGNOSE: The retrieval parameter is wrong.
# -----------------------------------------------------------

# ERROR 3: k=1 means we only retrieve ONE chunk. For a question
# that spans multiple meetings or topics, a single chunk is
# almost never sufficient. Typical k values are 3-5 for
# small corpora and 5-10 for larger ones.

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}  # BUG: only retrieves 1 chunk!
)

# Build the QA chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""Answer the question based on the context below.

Context: {context}

Question: {input}""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

#qa_chain = RetrievalQA.from_chain_type(
   # llm=llm,
   # chain_type="stuff",
    #retriever=retriever,
    #return_source_documents=True
#)

# Test with a question that requires multiple meetings
test_query = "How did the Fed's stance on inflation evolve throughout 2023?"
answer = qa_chain.invoke(test_query)

print(f'Question: {test_query}')
print(f'\nAnswer: {answer}')
print(f'\nPROBLEM: Only 1 source retrieved. This question requires context')
print('from multiple meetings to trace the evolution of Fed policy.')

Question: How did the Fed's stance on inflation evolve throughout 2023?

Answer: Based on the context provided, the Federal Open Market Committee (FOMC) in March 2023 likely indicated a continued focus on addressing inflation, which had been a significant concern for the Fed. Throughout 2023, the Fed's stance on inflation may have evolved as they assessed economic indicators, inflation rates, and the effectiveness of their monetary policy measures. If inflation remained high, the Fed might have continued to raise interest rates or maintain a hawkish stance to curb inflationary pressures. Conversely, if inflation showed signs of stabilizing or decreasing, the Fed could have considered a more dovish approach, potentially pausing rate hikes or signaling a shift in policy. Overall, the evolution of the Fed's stance on inflation throughout 2023 would have been influenced by ongoing economic data and the broader economic environment.

PROBLEM: Only 1 source retrieved. This question requires 

---

## YOUR TASK: Fix the RAG Pipeline

Correct all three errors and rebuild the pipeline from scratch.

**Verification checkpoints:**
- Chunks should average 400-600 characters with ~100 character overlap
- Embedding model should be `text-embedding-3-small`
- Retriever should return k=4 or k=5 documents
- Test query about Fed stance evolution should cite 3+ meetings

In [19]:
# -----------------------------------------------------------
# YOUR TASK — Fix all three errors and rebuild the pipeline
# -----------------------------------------------------------

# Fix 1: Correct chunking parameters
text_splitter_fixed = RecursiveCharacterTextSplitter(
    chunk_size=500,          # FILL IN: appropriate chunk size
    chunk_overlap=100,       # FILL IN: appropriate overlap
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_fixed = text_splitter_fixed.split_documents(fomc_documents)
print(f'Fixed chunks: {len(chunks_fixed)}')
print(f'Avg chunk length: {np.mean([len(c.page_content) for c in chunks_fixed]):.0f} chars')


# Fix 2: Correct embedding model
embeddings_fixed = OpenAIEmbeddings(
    model="text-embedding-3-small"              # FILL IN: current recommended model
)

vectorstore_fixed = Chroma.from_documents(
    documents=chunks_fixed,
    embedding=embeddings_fixed,
    collection_name="fomc_minutes_fixed"
)


# Fix 3: Correct retrieval parameter
retriever_fixed = vectorstore_fixed.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}  # FILL IN: appropriate k value
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain_fixed = (
    {"context": retriever_fixed | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Verification: re-run the same test query
answer_fixed = qa_chain_fixed.invoke(test_query)
print(f'\nFixed Answer: {answer_fixed}')

Fixed chunks: 10
Avg chunk length: 360 chars

Fixed Answer: Throughout 2023, the Federal Open Market Committee's (FOMC) stance on inflation evolved from a focus on high inflation levels to a more optimistic view of disinflation. 

In January 2023, the FOMC noted that inflation was well above their 2 percent objective, indicating a strong concern about persistent inflationary pressures. The labor market was described as very tight, with low unemployment and elevated wage growth, which contributed to the inflationary environment.

By September 2023, the FOMC acknowledged some improvements, citing supply-side factors that had contributed to a decline in inflation. However, they also recognized that labor market conditions, while easing somewhat, remained tight, and emphasized the need for careful communication regarding future policy decisions.

By December 2023, the FOMC reported that the disinflation process was continuing, suggesting a significant shift towards a more positive outlook 

---

## EXTEND: Query the Pipeline with 3 Economic Questions

Ask your fixed pipeline 3 questions about the 2023 FOMC minutes. For each question, evaluate:
1. **Retrieval quality** — Did the retriever find the right source documents?
2. **Answer grounding** — Is the answer supported by the retrieved text, or does it hallucinate?
3. **Completeness** — Does the answer address all parts of the question?

In [26]:
# -----------------------------------------------------------
# YOUR TASK — Query the fixed pipeline with 3 economic questions
# -----------------------------------------------------------

questions = [
    # Question 1: Should require info from multiple meetings
    "How did the Federal Reserve interest rate decisions change between January and December 2022?",  # FILL IN your question
    
    # Question 2: Should test specificity (a narrow, factual question)
    "What was the Federal Reserve target inflation rate in 2022?",  # FILL IN your question
    
    # Question 3: Should test reasoning across documents
    "What factors did the Fed weigh when deciding whether to raise or hold interest rates in 2023?",  # FILL IN your question
]
evaluations = [
    {"retrieval": 2, "grounding": 2, "completeness": 2, "notes": "Wrong year retrieved"},
    {"retrieval": 4, "grounding": 5, "completeness": 4, "notes": "Simple factual question answered well"},
    {"retrieval": 5, "grounding": 5, "completeness": 5, "notes": "Multi-document reasoning worked well"},
]
for i, q in enumerate(questions, 1):
    print(f'\n{"="*60}')
    print(f'Question {i}: {q}')
    print(f'{"="*60}')
    
    result = qa_chain_fixed.invoke(q)

    print(f'\nAnswer: {result}')
    print(f'\nSources: (retrieved via retriever)')
    
    # YOUR EVALUATION (fill in after running):
    
    eval = evaluations[i-1]
    print(f'\n--- Your Evaluation ---')
    print(f'Retrieval quality (1-5): {eval["retrieval"]}')
    print(f'Answer grounding (1-5):  {eval["grounding"]}')
    print(f'Completeness (1-5):      {eval["completeness"]}')
    print(f'Notes: {eval["notes"]}')


Question 1: How did the Federal Reserve interest rate decisions change between January and December 2022?

Answer: The context provided does not include specific information about the Federal Reserve's interest rate decisions between January and December 2022. However, it does indicate that by December 2023, the median projection for the federal funds rate at the end of 2024 was 4.6 percent, suggesting that rate cuts could begin in 2024. In contrast, the January 2023 meeting highlighted that inflation was well above the Committee's 2 percent objective, and the labor market was very tight, which typically suggests a need for higher interest rates to combat inflation. 

To summarize, while the specific changes in interest rate decisions between January and December 2022 are not detailed in the provided context, it can be inferred that the Federal Reserve was likely in a tightening phase in early 2023 due to high inflation, and by December 2023, there was a shift towards potentially lowe

---

## EXTEND: RAG vs. Direct LLM Comparison

For each of your 3 questions, also ask the LLM **without** retrieval (no source documents). Compare the answers to assess whether RAG grounding improves accuracy and reduces hallucination.

In [31]:
# -----------------------------------------------------------
# YOUR TASK — Compare RAG vs. direct LLM answers
# -----------------------------------------------------------

direct_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

comparisons = [
    {"specific": "Direct", "accurate": "Direct", "cites": "RAG", "hallucination": "Direct"},
    {"specific": "Direct", "accurate": "Same", "cites": "RAG", "hallucination": "Same"},
    {"specific": "RAG", "accurate": "RAG", "cites": "RAG", "hallucination": "Direct"},
]

for i, q in enumerate(questions, 1):
    print(f'\n{"="*60}')
    print(f'Question {i}: {q}')
    print(f'{"="*60}')
    
    # RAG answer (from fixed pipeline)
    rag_answer = qa_chain_fixed.invoke(q)
    
    # Direct LLM answer (no retrieval)
    direct_answer = direct_llm.invoke(q).content
    
    print(f'\nRAG Answer:')
    print(f'  {rag_answer[:300]}...' if len(rag_answer) > 300 else f'  {rag_answer}')
    
    print(f'\nDirect LLM Answer:')
    print(f'  {direct_answer[:300]}...' if len(direct_answer) > 300 else f'  {direct_answer}')
    
    comp = comparisons[i-1]
    print(f'\n--- Comparison ---')
    print(f'Which is more specific?    {comp["specific"]}')
    print(f'Which is more accurate?    {comp["accurate"]}')
    print(f'Which cites sources?       {comp["cites"]}')
    print(f'Risk of hallucination?     {comp["hallucination"]}')


Question 1: How did the Federal Reserve interest rate decisions change between January and December 2022?

RAG Answer:
  The context provided does not include specific information about the Federal Reserve's interest rate decisions between January and December 2022. However, it does indicate that by December 2023, the median projection for the federal funds rate at the end of 2024 was 4.6 percent, suggesting that rate...

Direct LLM Answer:
  Between January and December 2022, the Federal Reserve made significant changes to its interest rate policy in response to rising inflation. 

1. **January 2022**: The Federal Reserve maintained the federal funds rate at a range of 0% to 0.25%. At this time, inflation was already a concern, but the ...

--- Comparison ---
Which is more specific?    Direct
Which is more accurate?    Direct
Which cites sources?       RAG
Risk of hallucination?     Direct

Question 2: What was the Federal Reserve target inflation rate in 2022?

RAG Answer:
  The Fed

---

## Final Reflection

Answer each question in 2-3 sentences.

### Q1: When does RAG outperform direct LLM prompting? When does it not matter?

*Your answer:* RAG outperforms direct prompting when questions require specific, verifiable information from a defined document set like in Question 3, where RAG grounded its answer in actual FOMC meeting language. RAG does not add much value for simple factual questions with well-known answers (like the 2% inflation target) or when the relevant documents are not in the corpus, as with our 2022 question.



### Q2: How do chunking parameters affect retrieval quality?

*Connect to the bias-variance tradeoff from Ch 15 — small chunks have high variance (noisy retrieval), large chunks have high bias (diluted relevance).*

*Your answer:* Chunk size controls the bias-variance tradeoff in retrieval. Chunks of 50 characters produced fragments like "Meeting...." that were too small to carry meaning, causing noisy, low-quality retrieval. Larger chunks preserve coherence but risk diluting relevance by including off-topic content; overlap of 10-20% helps maintain context across chunk boundaries.



### Q3: In what economic research contexts would you use RAG over direct prompting?

*Think about: policy analysis on proprietary documents, literature review, regulatory compliance, central bank communication analysis.*

*Your answer:* RAG is preferable when analyzing proprietary or domain-specific documents such as central bank communications, SEC filings, or internal policy memos that fall outside an LLM's training data. It is also valuable for regulatory compliance and literature review tasks where answers must be traceable to specific source documents rather than general model knowledge.



---

## Submission

Submit the following on Canvas:

1. **Deployed portfolio URL** (Vercel link)
2. **This notebook** with all code cells run, evaluations filled in, and reflections answered
3. **GitHub repository link** for the portfolio source code

```bash
cd econ-portfolio
git add .
git commit -m "Lab 25: AI Literacy — Portfolio + RAG Pipeline"
git push origin main
```